# SICD — Semantic Interference & Cognitive Dissonance (camera-ready)

Stress-tests biomedical chain-of-thought reasoning under escalating semantic interference.

| Stage | Cell | What it does |
|-------|------|--------------|
| Setup | 0–4 | Fetch files (Colab), environment, API keys, imports, case overview |
| Generation | 5 | cases x 4 levels x temperatures {0.4, 0.7, 0.9} -> per-temp caches |
| Extraction + Scoring | 6 | UMLS concepts -> Density / Specificity / Oscillation / Regression / SDR |
| Evaluation | 7 | Spearman rho per temperature + ablation table + bootstrap CIs + per-case SDR |
| Visualization | 8 | Signal trajectories + SDR temperature-ablation overlay |

**Interference levels (harmonized prompts — identical across models):**
- Level 0 (Control): standard step-by-step diagnostic reasoning, no interference.
- Level 1 (Soft): "consider potential implications or differential considerations from the [interference domain] domain."
- Level 2 (Hard): "frame every step of your reasoning through the lens of [interference domain] pathophysiology and mechanisms."
- Level 3 (Full Dissonance): "explain why this presentation is actually a manifestation of [interference disease]."

**Oscillation note:** Oscillation is now *domain-frame* oscillation (target<->interference switching), reusing the SDR classifier. The earlier semantic-type oscillation was structurally pinned at 0 because the UMLS /search extraction never populates semantic types.


In [ ]:
# ── Colab: fetch experiment files (auto-skips if already present) ──────────────
# In Colab the support files (.py + utils/) are not in the session yet. This cell
# extracts a .zip you upload (or one already sitting in /content). It is skipped
# automatically when the files are present (e.g. running locally in VSCode, or on
# a re-run), so "Run all" never hangs.
import os, glob, zipfile

def _have_support_files():
    return os.path.exists('sicd_cases.py') or bool(
        glob.glob('/content/**/sicd_cases.py', recursive=True))

if not _have_support_files():
    for zp in glob.glob('/content/*.zip'):
        try:
            with zipfile.ZipFile(zp) as z:
                z.extractall('/content')
            print('Extracted', zp)
        except Exception as e:
            print('Could not extract', zp, ':', e)

if not _have_support_files():
    try:
        from google.colab import files
        print('Upload a .zip of the experiment folder '
              '(must contain sicd_cases.py, sicd_scorers.py, utils/).')
        for name in files.upload():
            if name.lower().endswith('.zip'):
                with zipfile.ZipFile(name) as z:
                    z.extractall('/content')
                print('Unzipped', name)
    except Exception as e:
        print('Upload step skipped (not on Colab?):', e)

print('Support files present:', _have_support_files())


In [ ]:
# ── 0. Environment setup ──────────────────────────────────────────────────────
import os, sys, json, time, shutil
from pathlib import Path

def setup_environment():
    found_root = None
    for root, dirs, files in os.walk('/content'):
        if 'sicd_cases.py' in files:
            found_root = root
            break
    if not found_root and os.path.isdir('/content') and 'cot_generator.py' in os.listdir('/content'):
        found_root = '/content'
    if not found_root and os.path.exists('sicd_cases.py'):
        found_root = os.getcwd()   # running locally (e.g. VSCode)

    if found_root:
        os.chdir(found_root)
        if found_root not in sys.path:
            sys.path.insert(0, found_root)
        # Fix flattened structure if necessary
        if not os.path.exists('utils'):
            os.makedirs('utils', exist_ok=True)
        utils_files = ['cot_generator.py', 'concept_extractor_v2.py', 'umls_api_linker.py',
                       'umls_checker.py', 'umls_density_scorer.py', 'umls_specificity_scorer.py',
                       'umls_oscillation_scorer.py', 'umls_regression_scorer.py', 'umls_local_db.py']
        for f in utils_files:
            if os.path.exists(f) and not os.path.isdir(f):
                shutil.move(f, os.path.join('utils', f))
        if not os.path.exists('utils/__init__.py'):
            open('utils/__init__.py', 'w').close()
        print(f'Environment configured at: {found_root}')
    else:
        print('WARNING: could not find sicd_cases.py. Run the fetch cell above (Colab) '
              'or open this notebook from its own folder (local).')

setup_environment()
if 'google.colab' in sys.modules:
    !pip install -q requests openai python-dotenv numpy scipy matplotlib pandas
print('Environment ready.')


In [ ]:
# ── 1. API Keys ───────────────────────────────────────────────────────────────
# Prompts securely (input is hidden) and skips any key already set in the env.
import os, getpass
for _k in ['OPENROUTER_API_KEY', 'UMLS_API_KEY']:
    if not os.environ.get(_k):
        os.environ[_k] = getpass.getpass(f'Enter {_k}: ')
print('API keys set:', {k: bool(os.environ.get(k)) for k in ['OPENROUTER_API_KEY', 'UMLS_API_KEY']})


In [ ]:
# ── 2. Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from utils.cot_generator import generate as cot_generate
from utils.concept_extractor_v2 import extract_concepts
from utils.umls_density_scorer import score_density
from utils.umls_specificity_scorer import score_specificity
from utils.umls_oscillation_scorer import score_oscillation
from utils.umls_regression_scorer import score_regression
from utils import umls_api_linker

from sicd_cases import CASES, INTERFERENCE_LEVELS, build_prompt
from sicd_cases import DOMAIN_SEMANTIC_TYPES, DOMAIN_KEYWORDS
from sicd_scorers import score_split_density

print('All modules imported.')
print(f'  UMLS configured: {umls_api_linker.is_configured()}')


In [ ]:
# ── 3. Case Overview ──────────────────────────────────────────────────────────
print(f'{len(CASES)} cases x 4 interference levels = {len(CASES)*4} chains')
print('-' * 80)
for c in CASES:
    print(f'{c["title"]:<28} | Target: {c["domain"]:<15} | Interf: {c["interference_domain"]}')


In [ ]:
# ── 4. CoT Generation — temperature ablation ──────────────────────────
# Re-run the full corpus at each temperature so the surrender/resistance pattern
# can be checked for robustness to decoding temperature (reviewer concern).
# 0.7 is the matched headline; 0.4/0.9 ablate.
MODEL         = 'anthropic/claude-haiku-4-5'
MODEL_TAG     = 'haiku'
TEMPS         = [0.4, 0.7, 0.9]
HEADLINE_TEMP = 0.7

Path('data').mkdir(exist_ok=True)

def generate_corpus(temp):
    """Generate (or load from cache) all cases x 4 levels at one temperature."""
    ttag = f"{temp:.1f}".replace('.', '')          # 0.4 -> '04'
    cache_path = Path(f'data/sicd_cache_{MODEL_TAG}_t{ttag}.json')
    if cache_path.exists():
        print(f'[T={temp}] loading cache {cache_path.name}')
        return json.loads(cache_path.read_text())
    total = len(CASES) * len(INTERFERENCE_LEVELS)
    done = 0
    print(f'[T={temp}] generating {total} chains with {MODEL} ...')
    raw = {}
    for case in CASES:
        raw[case['id']] = {}
        for label, ordinal in INTERFERENCE_LEVELS:
            prompt = build_prompt(case, label)
            result = cot_generate(prompt, prefer='openrouter', model=MODEL, temperature=temp)
            raw[case['id']][label] = result['steps']
            done += 1
            print(f'  [T={temp}] {done}/{total}  {case["title"]} [{label}] -> {len(result["steps"])} steps')
            time.sleep(0.5)
    cache_path.write_text(json.dumps(raw, indent=2))
    print(f'[T={temp}] cached -> {cache_path.name}')
    return raw

def build_chains(raw):
    chains = []
    for ci, case in enumerate(CASES):
        for label, ordinal in INTERFERENCE_LEVELS:
            chains.append({
                'id': f'{case["id"]}_{label}', 'label': f'{case["title"]} - {label}',
                'steps': raw[case['id']][label], 'case_idx': ci,
                'level_label': label, 'gt_ordinal': ordinal,
                'target_domain': case['domain'], 'interference_domain': case['interference_domain'],
            })
    return chains

CHAINS_BY_TEMP = {t: build_chains(generate_corpus(t)) for t in TEMPS}
print('Chains per temperature:', {t: len(v) for t, v in CHAINS_BY_TEMP.items()})


In [ ]:
# ── 5. Concept Extraction & Scoring (per temperature) ─────────────────────────
def score_chain(chain):
    chain['concepts']    = extract_concepts(chain['steps'], top_k=5)
    chain['density']     = score_density(chain['concepts'], steps=chain['steps'])
    chain['specificity'] = score_specificity(chain['concepts'])
    chain['oscillation'] = score_oscillation(
        chain['concepts'],
        target_domain=chain['target_domain'],
        interference_domain=chain['interference_domain'],
        domain_semantic_types=DOMAIN_SEMANTIC_TYPES,
        domain_keywords=DOMAIN_KEYWORDS,
    )
    chain['regression']  = score_regression(chain['concepts'])
    chain['split_density'] = score_split_density(
        chain['concepts'],
        target_domain=chain['target_domain'],
        interference_domain=chain['interference_domain'],
        domain_semantic_types=DOMAIN_SEMANTIC_TYPES,
        domain_keywords=DOMAIN_KEYWORDS,
        steps=chain['steps'],
    )
    return chain

for t, chains in CHAINS_BY_TEMP.items():
    print(f'[T={t}] extracting + scoring {len(chains)} chains ...')
    for i, chain in enumerate(chains):
        score_chain(chain)
        if (i + 1) % 10 == 0:
            print(f'    {i + 1}/{len(chains)}')
print('Scoring complete for all temperatures.')


In [ ]:
# ── 6. Spearman rho Evaluation + Temperature Ablation ─────────────────────────
SIGNALS = [
    ('Density slope',     'density',       'slope',              1),
    ('Specificity slope', 'specificity',   'depth_slope',        1),
    ('Oscillation',       'oscillation',   'oscillation_score',  1),
    ('Regression',        'regression',    'regression_score',   1),
    ('Split Ratio (SDR)', 'split_density', 'mean_density_ratio', -1),
]

def signal_value(ch, key, score_key):
    return ch[key].get(score_key, 0.0) or 0.0

def boot_ci(vals, gt, n=2000, seed=0):
    """95% bootstrap CI for Spearman rho."""
    rng = np.random.default_rng(seed)
    vals = np.asarray(vals, float); gt = np.asarray(gt, float); m = len(vals)
    stats = []
    for _ in range(n):
        idx = rng.integers(0, m, m)
        if len(np.unique(gt[idx])) < 2 or len(np.unique(vals[idx])) < 2:
            continue
        stats.append(spearmanr(vals[idx], gt[idx]).correlation)
    if not stats:
        return (float('nan'), float('nan'))
    return (float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5)))

abl_rows = []
for t in TEMPS:
    chains = CHAINS_BY_TEMP[t]
    gt = [ch['gt_ordinal'] for ch in chains]
    print(f'\n=== {MODEL_TAG} @ T={t}  (n={len(chains)}) ===')
    print(f'{"Signal":<20}{"rho":>8}{"p":>10}   95% CI')
    print('-' * 56)
    for name, key, sk, direction in SIGNALS:
        vals = [signal_value(ch, key, sk) for ch in chains]
        rho, p = spearmanr(vals, gt)
        lo, hi = boot_ci(vals, gt)
        print(f'{name:<20}{rho:>8.3f}{p:>10.4f}   [{lo:+.3f}, {hi:+.3f}]')
        abl_rows.append({'model': MODEL_TAG, 'temp': t, 'signal': name,
                         'rho': round(rho, 3), 'p': round(p, 4),
                         'ci_lo': round(lo, 3), 'ci_hi': round(hi, 3)})

abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv(f'data/sicd_ablation_{MODEL_TAG}.csv', index=False)
print(f'\nAblation table saved -> data/sicd_ablation_{MODEL_TAG}.csv')

sdr = abl_df[abl_df.signal == 'Split Ratio (SDR)'].set_index('temp')[['rho', 'p', 'ci_lo', 'ci_hi']]
print('\nSDR vs interference by temperature:'); print(sdr)

hl = CHAINS_BY_TEMP[HEADLINE_TEMP]
pc = {}
for ch in hl:
    pc.setdefault(ch['case_idx'], {})[ch['gt_ordinal']] = signal_value(ch, 'split_density', 'mean_density_ratio')
pc_df = pd.DataFrame(pc).T.sort_index()
pc_df.columns = ['Control', 'Soft', 'Hard', 'Dissonance'][:pc_df.shape[1]]
pc_df.index = [CASES[i]['title'] for i in pc_df.index]
print(f'\nPer-case SDR by level @ T={HEADLINE_TEMP}:'); print(pc_df.round(3))


In [ ]:
# ── 7. Visualization — Trajectories + SDR temperature ablation ────────────────
level_tick = ['Control', 'Soft', 'Hard', 'Dissonance']

hl = CHAINS_BY_TEMP[HEADLINE_TEMP]
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, key, sk, _d) in zip(axes, SIGNALS):
    means = [np.mean([signal_value(ch, key, sk) for ch in hl if ch['gt_ordinal'] == oi]) for oi in range(4)]
    ax.plot(range(4), means, 'k-s', lw=2, ms=8)
    ax.set_xticks(range(4)); ax.set_xticklabels(level_tick, fontsize=8)
    ax.set_title(f'{name} (T={HEADLINE_TEMP})'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f'data/sicd_trajectories_{MODEL_TAG}.png', dpi=150, bbox_inches='tight'); plt.show()

plt.figure(figsize=(6, 4))
for t in TEMPS:
    ch_t = CHAINS_BY_TEMP[t]
    means = [np.mean([signal_value(ch, 'split_density', 'mean_density_ratio') for ch in ch_t if ch['gt_ordinal'] == oi]) for oi in range(4)]
    plt.plot(range(4), means, '-o', label=f'T={t}')
plt.xticks(range(4), level_tick); plt.ylabel('Split Density Ratio')
plt.title(f'SDR vs interference - {MODEL_TAG}'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f'data/sicd_sdr_ablation_{MODEL_TAG}.png', dpi=150, bbox_inches='tight'); plt.show()
